# DLC Kinematics: Tongue

Full tongue kinematics pipeline:

- Phase 7a.2 — interpolated v5p3 traces
- Phase 7a.3 V3 — per-trial scalars (angles, traces, endpoints)
- Phase 7a.4 — angle/phase QC visuals (5 families per session)
- Phase 7a.5 — tongue/jaw mismatch classifier + within-session QC plots
- Phase 7a.6 — mean/SEM per-session families (individual_sessions, trial_overlays)
- Phase 7a.6 — mean/SEM cohort families (session_overlays, binned_overlays, mean_trace, mean_video)

Outputs go to `stroke_orofacial_pipeline_outputs/dlc_kinematics/.../tongue/...` on MICROSCOPE. Each per-phase loop handles per-session errors gracefully (try/except, log, continue) — matching the patterns in notebooks 10/11/12.

**Phase 7a.6 USER INPUTS extensions:** `WINDOWS_MS`, `MEAN_VIDEO_WINDOWS_SUBSET`, `ALLOW_GIF_FALLBACK`, `INCLUDE_MEAN_VIDEO`, `BIN_SIZES_DAYS`. See cell below for the C7.x learnings each surfaces. ffmpeg is required for `mean_video` MP4 emission (bundled in `environment.yml` as of Phase 7a.6 / C7.6).

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

# Animals to process. Default to PS46 only — kinematics + mean/SEM
# rendering is the slowest pipeline in this repo, and OVERWRITE="skip"
# below keeps reruns near-instant on cached outputs. Expand to the full
# cohort (PS46-PS55) for a fresh sweep.
ANIMALS = ["PS46"]

# Dates to process per animal. None = all dates in the animal's
# session range (from configs/animals.yaml).
DATES = None

# Overwrite mode: "skip" (default) or "force"
OVERWRITE = "skip"

# Phase 7a.6 — reward-aligned windows (ms, NOT seconds; matches the
# legacy rewardAligned_{pre}pre-{post}post/ directory convention).
WINDOWS_MS = [(100, 1000), (500, 5000)]

# Phase 7a.6 — mean_video windows (subset of WINDOWS_MS). Mirrors the
# C7.7 dlc_kinematics.mean_sem_visual.mean_video.mean_video_windows_subset
# config knob. Default: short window only — long window MP4 render
# time scales with duration and (-500/5000ms) was deemed not worth
# the analytical value (Priya 2026-05-02). Other plot families
# (individual_sessions, trial_overlays, session_overlays,
# binned_overlays, mean_trace) still emit for ALL WINDOWS_MS entries
# — only mean_video honors this allow-list.
MEAN_VIDEO_WINDOWS_SUBSET = [(100, 1000)]

# Phase 7a.6 — fail-loud on missing ffmpeg (C7.5 default). Set to True
# only if running in a dev/CI env where ffmpeg cannot be installed;
# mean_video will then degrade to GIF via PillowWriter (much larger
# files, posterized colors). Recovery for the default RuntimeError:
# `conda install -c conda-forge ffmpeg` in the stroke_orofacial env
# (or rebuild from environment.yml — ffmpeg is bundled there as of
# Phase 7a.6 / C7.6).
ALLOW_GIF_FALLBACK = False

# Phase 7a.6 — opt-out of mean_video rendering entirely (e.g., when
# iterating on per-trial scalars without needing the cohort
# animations). True = render MP4s; False = skip mean_video, all
# other 7a.6 families still emit.
INCLUDE_MEAN_VIDEO = True

# Phase 7a.6 — bin sizes (days) for mean_trace + mean_video. None =
# read animal_info.bin_days from configs/animals.yaml (default).
# Set explicitly to override (e.g., [2, 7] to emit both 2-day and
# 7-day binned plots for every animal).
BIN_SIZES_DAYS = None

BODYPART = "tongue"

In [ ]:
import datetime
import logging

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.dlc_kinematics.interpolation import (
    build_interpolated_trace_by_animal_date,
)
from stroke_orofacial.dlc_kinematics.mean_sem_visual import (
    build_mean_sem_by_animal_date,
    build_mean_sem_cohort,
)
from stroke_orofacial.dlc_kinematics.tongue_jaw_mismatch import (
    classify_tongue_jaw_mismatch_by_animal_date,
    render_within_session_figures,
)
from stroke_orofacial.dlc_kinematics.tongue_pertrial import (
    build_tongue_pertrial_by_animal_date,
)
from stroke_orofacial.dlc_kinematics.tongue_visual import (
    render_tongue_qc_figures,
)
from stroke_orofacial.manifest import build_manifest
from stroke_orofacial.paths import PathResolver

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()

In [ ]:
# ============================================================
# Build a SessionManifest per (animal, date). Surfaces DLC h5 +
# alignment-template discovery up-front so missing inputs fail at
# this step rather than partway through the per-phase loops below.
# ============================================================

manifests = []  # list of (animal_id, date, manifest)

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for date in dates:
        manifest = build_manifest(animal_id, date, resolver, info)
        manifests.append((animal_id, date, manifest))

n_with_dlc = sum(1 for _, _, m in manifests if m.dlc_h5_files)
n_with_align = sum(1 for _, _, m in manifests if m.alignment_template is not None)
print(f"Built {len(manifests)} manifests:")
print(f"  with DLC files:           {n_with_dlc}")
print(f"  with alignment template:  {n_with_align}")

In [ ]:
# ============================================================
# Phase 7a.2: interpolated v5p3 traces.
# ============================================================
n_ok, n_skip, n_err = 0, 0, 0
for animal_id, date, manifest in manifests:
    date_str = date.strftime("%Y%m%d")
    try:
        path = build_interpolated_trace_by_animal_date(
            animal_id, date, BODYPART, resolver, overwrite=OVERWRITE,
        )
        if path is None:
            n_skip += 1
        else:
            n_ok += 1
    except Exception as e:
        n_err += 1
        logging.warning("interp %s/%s: %s", animal_id, date_str, e)
print(f"Phase 7a.2 (interpolation, {BODYPART}): ok={n_ok} skip={n_skip} err={n_err}")

In [ ]:
# ============================================================
# Phase 7a.3 V3: tongue per-trial scalars (angles, traces, endpoints).
# ============================================================
n_ok, n_skip, n_err = 0, 0, 0
for animal_id, date, manifest in manifests:
    date_str = date.strftime("%Y%m%d")
    try:
        out = build_tongue_pertrial_by_animal_date(
            animal_id, date, resolver, overwrite=OVERWRITE,
        )
        if out is None:
            n_skip += 1
        else:
            n_ok += 1
    except Exception as e:
        n_err += 1
        logging.warning("pertrial %s/%s: %s", animal_id, date_str, e)
print(f"Phase 7a.3 (tongue per-trial scalars): ok={n_ok} skip={n_skip} err={n_err}")

In [ ]:
# ============================================================
# Phase 7a.4: tongue angle/phase QC visuals (5 families per session).
# ============================================================
n_ok, n_skip, n_err = 0, 0, 0
for animal_id, date, manifest in manifests:
    date_str = date.strftime("%Y%m%d")
    try:
        figs = render_tongue_qc_figures(
            animal_id, date, resolver, overwrite=OVERWRITE,
        )
        n_emitted = sum(len(p) for p in figs.values())
        if n_emitted == 0:
            n_skip += 1
        else:
            n_ok += 1
    except Exception as e:
        n_err += 1
        logging.warning("visual %s/%s: %s", animal_id, date_str, e)
print(f"Phase 7a.4 (angle/phase visual): ok={n_ok} skip={n_skip} err={n_err}")

In [ ]:
# ============================================================
# Phase 7a.5: tongue/jaw mismatch classifier + within-session QC plots.
# ============================================================
n_ok_class, n_skip_class, n_err_class = 0, 0, 0
n_ok_fig, n_skip_fig, n_err_fig = 0, 0, 0
for animal_id, date, manifest in manifests:
    date_str = date.strftime("%Y%m%d")
    try:
        path = classify_tongue_jaw_mismatch_by_animal_date(
            animal_id, date, resolver, overwrite=OVERWRITE,
        )
        if path is None:
            n_skip_class += 1
        else:
            n_ok_class += 1
    except Exception as e:
        n_err_class += 1
        logging.warning("mismatch class %s/%s: %s", animal_id, date_str, e)
        continue
    try:
        figs = render_within_session_figures(
            animal_id, date, resolver, overwrite=OVERWRITE,
        )
        if not any(figs.values()):
            n_skip_fig += 1
        else:
            n_ok_fig += 1
    except Exception as e:
        n_err_fig += 1
        logging.warning("mismatch figs %s/%s: %s", animal_id, date_str, e)
print(f"Phase 7a.5 (classifier): ok={n_ok_class} skip={n_skip_class} err={n_err_class}")
print(f"Phase 7a.5 (within-session figs): ok={n_ok_fig} skip={n_skip_fig} err={n_err_fig}")

In [ ]:
# ============================================================
# Phase 7a.6: per-(animal, date) per-session families
# (individual_sessions + trial_overlays) for every WINDOWS_MS entry.
# Cohort aggregates are cached so each (animal, window) is loaded once.
# ============================================================
n_ok, n_skip, n_err = 0, 0, 0
cohort_cache: dict = {}  # (animal -> {(pre_ms, post_ms): CohortAggregates})
for animal_id, date, manifest in manifests:
    date_str = date.strftime("%Y%m%d")
    try:
        per_animal_cache = cohort_cache.setdefault(animal_id, {})
        out = build_mean_sem_by_animal_date(
            BODYPART, animal_id, date, resolver,
            time_windows_ms=WINDOWS_MS,
            cohort_cache=per_animal_cache,
            overwrite=OVERWRITE,
        )
        n_emitted = sum(len(p) for p in out.values())
        if n_emitted == 0:
            n_skip += 1
        else:
            n_ok += 1
    except Exception as e:
        n_err += 1
        logging.warning(
            "mean_sem per-session %s/%s/%s: %s",
            animal_id, date_str, BODYPART, e,
        )
print(f"Phase 7a.6 (per-session, {BODYPART}): ok={n_ok} skip={n_skip} err={n_err}")

In [ ]:
# ============================================================
# Phase 7a.6: per-animal cohort families (session_overlays,
# binned_overlays, mean_trace, mean_video) across every WINDOWS_MS
# entry. mean_video is gated to MEAN_VIDEO_WINDOWS_SUBSET (C7.7) and
# fails loud on missing ffmpeg unless ALLOW_GIF_FALLBACK is True
# (C7.5). INCLUDE_MEAN_VIDEO=False skips mean_video entirely.
# ============================================================
n_ok, n_skip, n_err = 0, 0, 0
for animal_id in ANIMALS:
    try:
        out = build_mean_sem_cohort(
            BODYPART, animal_id, resolver,
            time_windows_ms=WINDOWS_MS,
            mean_video_windows=MEAN_VIDEO_WINDOWS_SUBSET,
            include_mean_video=INCLUDE_MEAN_VIDEO,
            allow_gif_fallback=ALLOW_GIF_FALLBACK,
            bin_sizes_days=BIN_SIZES_DAYS,
            overwrite=OVERWRITE,
        )
        n_emitted = sum(len(p) for p in out.values())
        if n_emitted == 0:
            n_skip += 1
        else:
            n_ok += 1
            print(f"  {animal_id}: " + ", ".join(
                f"{k}={len(v)}" for k, v in out.items()
            ))
    except Exception as e:
        n_err += 1
        logging.warning("mean_sem cohort %s/%s: %s", animal_id, BODYPART, e)
print(f"Phase 7a.6 (cohort, {BODYPART}): ok={n_ok} skip={n_skip} err={n_err}")

In [ ]:
print("Phase 7a.6 tongue pipeline complete.")
print(f"  animals processed: {len(ANIMALS)}")
print(f"  manifest pairs:    {len(manifests)}")
print(f"  windows:           {WINDOWS_MS}")
print(
    f"  mean_video gated:  "
    f"{MEAN_VIDEO_WINDOWS_SUBSET if INCLUDE_MEAN_VIDEO else 'OFF (INCLUDE_MEAN_VIDEO=False)'}"
)